Grundregeln:
- jede python Klasse hat eine übergeordnete Aufgabe
- jede python Klasse hält nur einen data file
- jede python Klasse importiert nur dann andere Klassen, wenn sie nur durch diese Importe als Gesamtes funktionieren (das ist in den meisten Fällen der DataManager). Wenn eine andere Klasse kein inhärenter Teil ist, wird sie nur als Funktions-Input für die Methode nutzbar gemacht.
- kein cleaning! Statt stillem Bereinigen stoppt die Funktion bei Ungereimtheiten und zeigt eine Fehlermeldung an

Zu adressierende Probleme:
- predicate requisite kleiner Teil der RO-Relationen. Reicht der aus?
- nur Abstracts in pubmed. Volltext-Reviews / Metaanalysen oder Bücher besser?
- manche Attribute sind bei fast jeder Entität TRUE. Sollte man diese entfernen? >90% TRUE?
- Scalability probleme vor allem der Normalisierung und fill_nan
- Vektordatenbank statt Pubmed_Abstracts_df besser?




Adressierte Probleme:
- causally_related ist oft falsch verknüpft. Aus der Liste der Prädikate entfernen? Antwort: Entfernt worden
- bis LLMClient().extract_entities werden nur Attribute ausgefüllt, die TRUE sind. Das ist zu ungenau. Von Anfang an müssen Attribute auch False gelabelt werden. Antwort: Extract entities labelt auch Falsche Entitäten
- mehrere Klassen haben durch das LLM denselben Namen bekommen, was verboten ist. Antwort: Prompt ist angepasst worden
- Abstracts enthalten sehr viele Entitäten, die nichts mit Wunden zu tun haben. Antwort: LLM prompt wurde so angepasst, dass nur domänenspezifische Entitäten integriert werden.
- Die formale Konzeptanalyse hält sich nicht an die BFO/OBO-Architektur. Antwort: attributes haben die form predicate: target. präformierte attributes sind die bfo_classes als is_a:bfo_class um die Einteilung in die BFo-Architektur zu erzwingen.
- es kann mehrere synonme Objekte oder Attribute geben. Antwort: objects und attributes werden per LLM zu standardbegriffen gemappt, dann grouping durch hard code
- Klassennamen werden aus Attributen gebildet. Antwort: Semantische Klassennamen kreiert LLM
- zu viele concepts und attribute? unschaffbar für hard code? erweiterter RAM und TPU
- zu viele Anfragen an LLM? Antwort: batching
- zu komplexe Anfragen an LLM? Antwort: Lange Frage besser als lange Antwort! (gemini 3 flash schafft 1mio tokens)
- komplette Nachverfolgbarkeit für selektives Löschen query-weise oder datumsweise bei schlechten einzelnen Ergebnissen: Antwort: meta_ids
- kein überladen des Speicher: Antwort: immer nur 10 dateien eines Präfix im Speicher und purge method des DataManager()
- LLM-Anfragen dauern zu lange, teils nur eine Anfrage pro 2,5 Minuten. Antwort: asyncio parallel prompting


# Setup

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import sys
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
# Works whether VS Code runs the notebook from project root or notebooks/
_cwd = Path().resolve()
_project_root = _cwd if (_cwd / "config").exists() else _cwd.parent
sys.path.insert(0, str(_project_root))
load_dotenv(_project_root / ".env")

from config.settings import (
    BASE_PATH,
    OWL_OUTPUT_FILE,
    ENTREZ_EMAIL,
    GOOGLE_API_KEY,
    BATCH_SIZE,
    PUBMED_SEARCH_TERMS,
    ALLOWED_PREDICATES,
)
print(f"Project root: {_project_root}")
print(f"ALLOWED_PREDICATES: {ALLOWED_PREDICATES}")

In [ ]:
from wound_ontology.orchestrator import Orchestrator

# Pipeline

## 1. Initialise

In [ ]:
pipeline = Orchestrator()

## 2. Fetch abstracts

In [ ]:
pipeline.run_pubmed_search(PUBMED_SEARCH_TERMS)

## 3. Extract entities → ConceptStore

In [ ]:
pipeline.run_extract_to_store(batch_size=BATCH_SIZE)

## 4. Build hierarchies (normalize + class / mereological)

In [ ]:
pipeline.run_build_hierarchies()

## 5. Fill missing entity–attribute pairs

In [ ]:
pipeline.run_store_fill()

## 6. Formal Concept Analysis

In [ ]:
pipeline.run_fca(
    subtree_root="Wound",
    predicates=["has_quality", "occurs_in", "caused_by", "is_treatment_for"],
)

## 7. Export → OWL

In [ ]:
pipeline.export_to_owl()

## 8. Enrich OWL (IRI renaming, BFO/RO import, UMLS synonyms)

In [ ]:
pipeline.run_enrich_owl(
    input_file=OWL_OUTPUT_FILE,
    output_file=str(Path(OWL_OUTPUT_FILE).parent / "ontology_enriched.owl"),
    import_bfo=True,
    import_ro=True,
    # umls_api_key=os.getenv("UMLS_API_KEY"),  # uncomment when key is available
)

# Inspection

In [ ]:
pipeline.lattice.summary()

# Maintenance

> Run these cells selectively — they modify or delete cached data.

In [ ]:
# Purge a specific cache prefix and re-run from that step
# pipeline.dm.purge_by_prefix("context_matrix", confirm=True)